In [21]:
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("..") / "data" / "processed"
train = pd.read_csv(DATA_DIR / "train.csv")
val = pd.read_csv(DATA_DIR / "val.csv")
test_pub = pd.read_csv(DATA_DIR / "test_pub.csv")

LABEL_NAMES = {0: "Background", 1: "Basis", 2: "Discuss", 3: "Differ", 4: "Support"}
print(train.shape, val.shape, test_pub.shape)

(1866, 3) (330, 3) (550, 3)


In [23]:
ZERO_SHOT_PROMPT = """You are an expert annotator for citation intent classification in Turkish scientific literature, following the Web of Science citation function taxonomy.

Classify the citation context below into exactly one of these 5 classes. The citation being classified is marked with <CITE>.

0 = Background: previously published research that orients the current study in its field; general/introductory context with no direct comparison to the current study's own results.
1 = Basis: datasets, methods, concepts, or tools that the authors directly use in their own study.
2 = Discuss: general discussion of a cited work that does not clearly fit any other class; a catch-all category.
3 = Differ: direct comparison between the current study's results and the cited work, where the results disagree or differ.
4 = Support: direct comparison between the current study's results and the cited work, where the results are confirmed/consistent.

Rules:
- Base your decision only on the given text.
- If a sentence merely mentions a reference without any explicit comparison to the current study's own results, prefer Background or Discuss over Support/Differ.
- Respond with only the numeric label (0-4).

Text to classify:
{text}"""

In [24]:
FEW_SHOT_PROMPT = """You are an expert annotator for citation intent classification in Turkish scientific literature, following the Web of Science citation function taxonomy.
Classify the citation context below into exactly one of these 5 classes. The citation being classified is marked with <CITE>.
0 = Background: previously published research that orients the current study in its field; general/introductory context with no direct comparison to the current study's own results.
1 = Basis: datasets, methods, concepts, or tools that the authors directly use in their own study.
2 = Discuss: general discussion of a cited work that does not clearly fit any other class; a catch-all category.
3 = Differ: direct comparison between the current study's results and the cited work, where the results disagree or differ.
4 = Support: direct comparison between the current study's results and the cited work, where the results are confirmed/consistent.
Labeled examples:
[Label 0] "Liu ve diğerleri <CITE>, iyileştirilmiş bir U-Net ağı tasarlamışlardır."
[Label 1] "Akciğer kanserinin tespiti için yapılan bu çalışmada, Kaggle platformunda açık erişimli popüler akciğer BT tarama görüntülerinden oluşan bir veri seti kullanılmıştır <CITE> ."
[Label 2] "Toplanan bu veriler MySQL, Microsoft SQL(MSSQL) <CITE> gibi istenilen bir veri tabanına yüklenebileceği gibi CSV (Virgülle Ayrılan Değerler) metin dosyası veya Excel dosyasına taşınabilir."
[Label 3] "Çalışma <CITE> ve bulut bilişimin sağlık hizmetlerinde kullanılmasını önermekle beraber mobil teknolojisinin kullanımı bu çalışmalarda yer almamaktadır."
[Label 4] "DVM kullanan benzer çalışmaların <CITE> yüksek doğrulukta sonuçlar verdiği görülmüştür."
Rules:
- Base your decision only on the given text.
- If a sentence merely mentions a reference without any explicit comparison to the current study's own results, prefer Background or Discuss over Support/Differ.
- Respond with only the numeric label (0-4).
Text to classify:
{text}"""

#### This notebook is for training LLMs

In [25]:
# from openai import OpenAI
# client = OpenAI()

# response = client.responses.create(
#     model="gpt-5.6",
#     input="Write a one-sentence bedtime story about a unicorn."
# )

# print(response.output_text)

In [26]:
from pydantic import BaseModel
from openai import OpenAI
import time

class CitationPrediction(BaseModel):
    label: int

client = OpenAI()

In [27]:
def classify_single(text, prompt_template=ZERO_SHOT_PROMPT, max_retries=3):
    prompt = prompt_template.format(text=text)
    for attempt in range(max_retries):
        try:
            response = client.responses.parse(
                model="gpt-5.6",
                input=prompt,
                text_format=CitationPrediction,
            )
            return response.output_parsed.label
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return 0

In [28]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def classify_dataset(df, prompt_template=ZERO_SHOT_PROMPT, max_workers=8):
    texts = df["citation_context"].tolist()
    preds = [None] * len(texts)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(classify_single, t, prompt_template): i for i, t in enumerate(texts)}
        done = 0
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            preds[idx] = future.result()
            done += 1
            if done % 20 == 0:
                print(f"Processed {done}/{len(texts)}")
    return preds

In [30]:
val["zero_shot_pred"] = classify_dataset(val, prompt_template=ZERO_SHOT_PROMPT)

Processed 20/330
Processed 40/330
Processed 60/330
Processed 80/330
Processed 100/330
Processed 120/330
Processed 140/330
Processed 160/330
Processed 180/330
Processed 200/330
Processed 220/330
Processed 240/330
Processed 260/330
Processed 280/330
Processed 300/330
Processed 320/330


In [31]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Zero-shot val Accuracy:", accuracy_score(val["citation_intent"], val["zero_shot_pred"]))
print("Zero-shot val Macro F1:", f1_score(val["citation_intent"], val["zero_shot_pred"], average="macro"))
print(classification_report(val["citation_intent"], val["zero_shot_pred"], target_names=list(LABEL_NAMES.values()), digits=3))

Zero-shot val Accuracy: 0.8272727272727273
Zero-shot val Macro F1: 0.6287817802745939
              precision    recall  f1-score   support

  Background      0.900     0.914     0.907       245
       Basis      0.708     0.829     0.764        41
     Discuss      0.167     0.136     0.150        22
      Differ      0.714     0.714     0.714         7
     Support      0.875     0.467     0.609        15

    accuracy                          0.827       330
   macro avg      0.673     0.612     0.629       330
weighted avg      0.822     0.827     0.821       330



In [32]:
val["few_shot_pred"] = classify_dataset(val, prompt_template=FEW_SHOT_PROMPT)

print("Few-shot val Accuracy:", accuracy_score(val["citation_intent"], val["few_shot_pred"]))
print("Few-shot val Macro F1:", f1_score(val["citation_intent"], val["few_shot_pred"], average="macro"))
print(classification_report(val["citation_intent"], val["few_shot_pred"], target_names=list(LABEL_NAMES.values()), digits=3))

Processed 20/330
Processed 40/330
Processed 60/330
Processed 80/330
Processed 100/330
Processed 120/330
Processed 140/330
Processed 160/330
Processed 180/330
Processed 200/330
Processed 220/330
Processed 240/330
Processed 260/330
Processed 280/330
Processed 300/330
Processed 320/330
Few-shot val Accuracy: 0.8272727272727273
Few-shot val Macro F1: 0.607346822742475
              precision    recall  f1-score   support

  Background      0.886     0.922     0.904       245
       Basis      0.680     0.829     0.747        41
     Discuss      0.100     0.045     0.062        22
      Differ      0.714     0.714     0.714         7
     Support      0.875     0.467     0.609        15

    accuracy                          0.827       330
   macro avg      0.651     0.596     0.607       330
weighted avg      0.804     0.827     0.811       330



In [33]:
test_pub["zero_shot_pred"] = classify_dataset(test_pub, prompt_template=ZERO_SHOT_PROMPT)

print("Zero-shot test_pub Accuracy:", accuracy_score(test_pub["citation_intent"], test_pub["zero_shot_pred"]))
print("Zero-shot test_pub Macro F1:", f1_score(test_pub["citation_intent"], test_pub["zero_shot_pred"], average="macro"))
print(classification_report(test_pub["citation_intent"], test_pub["zero_shot_pred"], target_names=list(LABEL_NAMES.values()), digits=3))

Processed 20/550
Processed 40/550
Processed 60/550
Processed 80/550
Processed 100/550
Processed 120/550
Processed 140/550
Processed 160/550
Processed 180/550
Processed 200/550
Processed 220/550
Processed 240/550
Processed 260/550
Processed 280/550
Processed 300/550
Processed 320/550
Processed 340/550
Processed 360/550
Processed 380/550
Processed 400/550
Processed 420/550
Processed 440/550
Processed 460/550
Processed 480/550
Processed 500/550
Processed 520/550
Processed 540/550
Zero-shot test_pub Accuracy: 0.8254545454545454
Zero-shot test_pub Macro F1: 0.6415604745514887
              precision    recall  f1-score   support

  Background      0.922     0.876     0.899       420
       Basis      0.733     0.892     0.805        74
     Discuss      0.136     0.200     0.162        30
      Differ      0.889     0.800     0.842        10
     Support      0.750     0.375     0.500        16

    accuracy                          0.825       550
   macro avg      0.686     0.629     0.64

In [34]:
test_final = pd.read_csv(Path("..") / "data" / "processed" / "test_final.csv")

test_final["zero_shot_pred"] = classify_dataset(test_final, prompt_template=ZERO_SHOT_PROMPT)

Processed 20/326
Processed 40/326
Processed 60/326
Processed 80/326
Processed 100/326
Processed 120/326
Processed 140/326
Processed 160/326
Processed 180/326
Processed 200/326
Processed 220/326
Processed 240/326
Processed 260/326
Processed 280/326
Processed 300/326
Processed 320/326


In [43]:
submission_llm = pd.DataFrame({
    "id": test_final["id"],
    "citation_intent": test_final["zero_shot_pred"],
})

RESULTS_DIR = Path("..") / "results"
RESULTS_DIR.mkdir(exist_ok=True)
submission_llm.to_csv(RESULTS_DIR / "submission_gpt56_zeroshot.csv", index=False)

print(submission_llm.shape)
print(submission_llm["citation_intent"].value_counts(normalize=True).sort_index())
print(submission_llm.head())

(326, 2)
citation_intent
0    0.674847
1    0.199387
2    0.079755
3    0.024540
4    0.021472
Name: proportion, dtype: float64
      id  citation_intent
0  29934                0
1  29935                0
2  29936                0
3  29937                0
4  29938                0


In [44]:
submission_llm = pd.concat([
    submission_llm,
    pd.DataFrame({"id": [46031], "citation_intent": [0]})
], ignore_index=True)

submission_llm.to_csv(RESULTS_DIR / "submission_gpt56_zeroshot.csv", index=False)
print(submission_llm.shape)

(327, 2)
